# Sprint 2 — Preparação de Dados para o LLM

Projeto Integrador — Construção de um Large Language Model (LLM) From Scratch

Baseado no Capítulo 2 (*Working with Text Data*) do livro **Build a Large Language Model (From Scratch)**, de Sebastian Raschka.

Este notebook reúne, em sequência, todos os componentes desenvolvidos na Sprint 2:
tokenização, vocabulário e Token IDs, Byte Pair Encoding (BPE), preparação de sequências,
embeddings, positional embeddings e DataLoader — reproduzindo o pipeline completo:

`Texto → Tokenização → Token IDs → Sequências → Embeddings → Positional Embeddings → Lote de dados → Entrada do modelo`


In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd().resolve().parents[0]  # ajuste se necessario: pasta raiz do repositorio
CAMINHO_CORPUS = BASE_DIR / "data" / "the-verdict.txt"

with open(CAMINHO_CORPUS, "r", encoding="utf-8") as f:
    texto_completo = f.read()

print("Total de caracteres no corpus:", len(texto_completo))
print(texto_completo[:200])


## 3.1 — Tokenização

Divide o texto bruto em tokens (palavras e sinais de pontuação), usando uma expressão
regular que separa vírgula, ponto, dois-pontos, ponto e vírgula, interrogação, exclamação,
aspas, parênteses, apóstrofo, travessão duplo (`--`) e espaços — mantendo cada delimitador
como um token próprio.


In [ ]:
import re


def tokenizar(texto):
    partes = re.split(r'([,.:;?_!"()\']|--|\s)', texto)
    return [p.strip() for p in partes if p.strip()]


# testes com frases diferentes
frases_teste = [
    "Teste do texto, trevizol e pedro! os cara topzera.",
    "Teste de travessão ---",
    "O Pedro é o mais lindo do grupo? Logico que sim!",
]

for frase in frases_teste:
    print(frase, "->", tokenizar(frase))


In [ ]:
# aplicando no corpus completo
tokens = tokenizar(texto_completo)
print("Total de tokens (regex) no corpus:", len(tokens))
print("Primeiros 30 tokens:", tokens[:30])


## 3.2 — Vocabulário e Token IDs

Constrói o vocabulário a partir dos tokens únicos do corpus (ordem alfabética), e cria as
funções `encode`/`decode` para converter texto em Token IDs e vice-versa. Também trata
tokens especiais (`<|unk|>` para palavras fora do vocabulário e `<|endoftext|>` para marcar
fim de texto).


In [ ]:
palavras_unicas = sorted(set(tokens))
print("Tamanho do vocabulário (sem tokens especiais):", len(palavras_unicas))
print("Primeiras 10 palavras únicas:", palavras_unicas[:10])


In [ ]:
tokens_especiais = ["<|unk|>", "<|endoftext|>"]
vocab_estendido = palavras_unicas + tokens_especiais
vocab = {token: id for id, token in enumerate(vocab_estendido)}

print("Novo tamanho do vocabulário:", len(vocab))
print("ID de <|unk|>:", vocab["<|unk|>"])
print("ID de <|endoftext|>:", vocab["<|endoftext|>"])


def encode(texto, vocab):
    tks = tokenizar(texto)
    return [vocab[t] if t in vocab else vocab["<|unk|>"] for t in tks]


def decode(ids, vocab):
    vocab_inverso = {id: token for token, id in vocab.items()}
    texto = " ".join(vocab_inverso[id] for id in ids)
    return re.sub(r'\s+([,.?!"()\'])', r'\1', texto)


# teste com frase que existe no corpus
frase = "It was not that my hostess was interesting"
ids = encode(frase, vocab)
print("\nFrase:", frase)
print("Token IDs:", ids)
print("Decodificado:", decode(ids, vocab))

# teste com frase fora do vocabulário
frase_fora = "Trevizol e pedro topzera demais"
ids_fora = encode(frase_fora, vocab)
print("\nFrase fora do vocabulário:", frase_fora)
print("Token IDs:", ids_fora)
print("Decodificado:", decode(ids_fora, vocab))


## 3.3 — Byte Pair Encoding (BPE) e preparação das sequências

O vocabulário construído no 3.2 é fechado: qualquer palavra fora do `the-verdict.txt` vira
`<|unk|>`. A partir daqui, seguindo o Capítulo 2 (seção 2.5), trocamos para o tokenizador
BPE do GPT-2 (`tiktoken`), que resolve esse problema quebrando palavras desconhecidas em
subpalavras conhecidas.


In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

# mesma frase que virou so <|unk|> no 3.2
texto_teste = "Trevizol e pedro topzera demais"
ids_bpe = tokenizer.encode(texto_teste)
print("Token IDs (BPE):", ids_bpe)
print("Decodificado:", tokenizer.decode(ids_bpe))


In [ ]:
enc_text = tokenizer.encode(texto_completo)
print("Total de tokens (BPE) no corpus:", len(enc_text))

enc_sample = enc_text[50:]
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print("Entrada (x):", x)
print("Alvo    (y):", y)

print()
for i in range(1, context_size + 1):
    contexto = enc_sample[:i]
    esperado = enc_sample[i]
    print(contexto, "---->", esperado)

print()
for i in range(1, context_size + 1):
    contexto = enc_sample[:i]
    esperado = enc_sample[i]
    print(tokenizer.decode(contexto), "---->", tokenizer.decode([esperado]))


In [ ]:
def gerar_pares(token_ids, max_length, stride):
    entradas = []
    alvos = []
    for i in range(0, len(token_ids) - max_length, stride):
        entradas.append(token_ids[i:i + max_length])
        alvos.append(token_ids[i + 1:i + max_length + 1])
    return entradas, alvos


entradas, alvos = gerar_pares(enc_text, max_length=4, stride=1)
print("Total de amostras geradas:", len(entradas))
print("Primeiras 3 amostras:")
for i in range(3):
    print(f"  entrada: {entradas[i]}  ->  alvo: {alvos[i]}")


## 3.4 — Embeddings

Transforma os Token IDs (números soltos, sem significado semântico) em vetores densos
treináveis, usando a camada `nn.Embedding` do PyTorch — uma tabela de consulta onde cada
linha é o vetor de um Token ID.


In [ ]:
import torch

torch.manual_seed(123)

vocab_size = 50257   # vocabulario do GPT-2 (BPE)
output_dim = 256      # dimensao escolhida para cada vetor de embedding

camada_embedding = torch.nn.Embedding(vocab_size, output_dim)
print("Formato da tabela de embeddings:", camada_embedding.weight.shape)


In [ ]:
entrada_exemplo = torch.tensor(entradas[0])
token_embeddings = camada_embedding(entrada_exemplo)

print("Token IDs de entrada:", entrada_exemplo)
print("Formato dos vetores gerados:", token_embeddings.shape)
print("Vetor do primeiro token:", token_embeddings[0])


## 3.5 — Positional Embeddings

Como o Transformer processa a sequência inteira de uma vez (sem noção nativa de ordem), é
preciso somar aos embeddings de token uma informação de posição — outra camada
`nn.Embedding`, agora indexada por posição em vez de por token.


In [ ]:
context_length = 4

camada_posicional = torch.nn.Embedding(context_length, output_dim)

posicoes = torch.arange(context_length)
pos_embeddings = camada_posicional(posicoes)

print("Posições:", posicoes)
print("Formato dos embeddings posicionais:", pos_embeddings.shape)


In [ ]:
input_embeddings = token_embeddings + pos_embeddings

print("Formato token_embeddings:", token_embeddings.shape)
print("Formato pos_embeddings:", pos_embeddings.shape)
print("Formato final (token + posicional):", input_embeddings.shape)


## 3.6 — DataLoader

Formaliza a geração de pares entrada/alvo (que já fizemos manualmente com `gerar_pares` no
3.3) usando as ferramentas oficiais do PyTorch: `Dataset` e `DataLoader`. É o que permite
gerar lotes (batches) de amostras para o treinamento.


In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


dataset_teste = GPTDatasetV1(texto_completo, tokenizer, max_length=4, stride=1)
print("Total de amostras no dataset:", len(dataset_teste))
print("Primeira amostra (entrada, alvo):", dataset_teste[0])


In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                          stride=128, shuffle=True, drop_last=True,
                          num_workers=0):
    tk = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tk, max_length, stride)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        drop_last=drop_last, num_workers=num_workers
    )
    return dataloader


dataloader = create_dataloader_v1(
    texto_completo, batch_size=8, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
entradas_lote, alvos_lote = next(data_iter)

print("Formato das entradas (lote):", entradas_lote.shape)
print("Formato dos alvos (lote):", alvos_lote.shape)
print("\nPrimeiro lote de entradas:\n", entradas_lote)


### Pipeline completo — do lote real aos embeddings finais

Fecha o fluxo da Sprint 2: pega um lote de verdade produzido pelo `DataLoader` e passa
pelas camadas de embedding de token e posicional, chegando na entrada final que alimentará
o mecanismo de atenção na Sprint 3.


In [ ]:
torch.manual_seed(123)

camada_embedding_final = torch.nn.Embedding(vocab_size, output_dim)
camada_posicional_final = torch.nn.Embedding(context_length, output_dim)

token_embeddings_lote = camada_embedding_final(entradas_lote)
pos_embeddings_lote = camada_posicional_final(torch.arange(context_length))

input_embeddings_lote = token_embeddings_lote + pos_embeddings_lote

print("Formato token_embeddings (lote):", token_embeddings_lote.shape)
print("Formato pos_embeddings:", pos_embeddings_lote.shape)
print("Formato final (pronto para o Transformer):", input_embeddings_lote.shape)
